# Huggingface Dataset Loader with GRIT

In [ ]:
from datasets import load_dataset
import json
from pathlib import Path

ds = load_dataset("zzliang/GRIT", split="train")

In [ ]:
ds[0]

In [ ]:
from pathlib import Path
import json
import requests
from tqdm import tqdm
from time import sleep

folder = Path("../GRIT")
image_folder = folder / "images"
image_folder.mkdir(parents=True, exist_ok=True)

json_path = folder / "grit.json"

# 假設 ds 已經是你的原始 dataset（HF Dataset 或 list[dict]）
# 1) 收集 URL
url_set = set()

for data in tqdm(ds, desc="collecting URLs"):
    url = data["url"]
    url_set.add(url)

print(f"共 {len(url_set)} 個獨立 image URL")

In [6]:
from pathlib import Path
from tqdm import tqdm
import requests

url2fname = {}
width = 8
sorted_urls = sorted(url_set)
image_folder = Path(image_folder)
image_folder.mkdir(parents=True, exist_ok=True)

# 校網環境可以設比較兇一點
timeout = (2, 5)  # (connect=2s, read=5s)

session = requests.Session()  # 重用連線，稍微快一點

for idx in tqdm(range(len(sorted_urls)), desc="downloading images", mininterval=0.1):
    url = sorted_urls[idx]
    fname = f"{idx+1:0{width}d}.jpg"
    out_path = image_folder / fname

    # resume：已存在直接略過
    if out_path.exists():
        url2fname[url] = fname
        continue

    try:
        r = session.get(url, timeout=timeout, stream=True)
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=16384):
                if not chunk:
                    continue
                f.write(chunk)
        url2fname[url] = fname
    except Exception:
        # 失敗就略過，不記 url2fname
        continue


downloading images:   0%|          | 3671/19190135 [26:01<2267:09:27,  2.35it/s] 


KeyboardInterrupt: 

In [ ]:
print(f"成功對應 {len(url2fname)} 個 URL -> 檔名")
# 3) 重組資料
new_data = []
missing = 0

for data in ds:
    url = data["url"]
    if url not in url2fname:
        missing += 1
        continue

    item = {
        "id": data["key"],           # 你剛剛說只留一個 id，這樣就好
        "caption": data["caption"],
        "images": [url2fname[url]],  # 跟訓練 schema 的 images 對位
        "ref_exps": data["ref_exps"],
    }
    new_data.append(item)

print(f"有效樣本 {len(new_data)} 筆，缺圖 {missing} 筆")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False)

print(f"{json_path.name} 包含 {len(new_data)} 筆資料")
print(new_data[0])